??Spark???ALS????????,??????demo

In [ ]:
from pyspark import SparkContext, SparkConf # ???????
from pyspark.mllib.recommendation import ALS
from math import sqrt
from operator import add
import itertools

# ??????????????????

# ????
def initSpark(): pass  # ???Spark???
def parseData(sparkContext, ratingsFile, moviesFile): return (None, None, None, None)  # ???????
def trainModel(trainingData, validationData, testData, iterations=5, lambda_=0.01, blocks=-1): pass  # ????????
def predict(model, moviesData, rating, user_id): pass  # ????????????????


# ????????
sc = initSpark()  # ???Spark???
trainingData, validationData, testData, moviesData = parseData(sc, "file:///root/notebook/data/ratings.dat",
                                                               "file:///root/notebook/data/movies.dat")  # ???????
model = trainModel(trainingData, validationData, testData)  # ??????
predict(model, moviesData, testData, 1)  # ????????????????
# sc.stop()  # ????Spark???

In [ ]:
# ???Spark???
# local???????,????????http://spark.apache.org/docs/latest/cluster-overview.html
def initSpark():
    conf = SparkConf().setAppName("CF").setMaster("local")
    sc = SparkContext(conf=conf)
    print ("init complete : sc = ", sc)
    return sc

In [ ]:
def parseData(sparkContext, ratingsFile, moviesFile):
    # ????,?????:user_id::movies_id::rating::time
    ratings = sparkContext.textFile(ratingsFile).map(lambda line: line.strip().split("::"))
    print("data.count() = %d" % ratings.count())
    # ???????????movieId::movieTitle
    movies = sparkContext.textFile(moviesFile).map(lambda line: line.strip().split("::"))

    # ?????,??????????????????????(60%), ?????(20%), ????(20%)
    ratingsData = ratings.map(lambda fields: (long(fields[3]) % 10, (int(fields[0]), int(fields[1]), float(fields[2]))))
    trainingData = ratingsData.filter(lambda x: x[0] < 6).values()
    validationData = ratingsData.filter(lambda x: x[0] >= 6 and x[0] < 8).values()
    testData = ratingsData.filter(lambda x: x[0] >= 8).values()

    print("training.count()=%d,validation.count()=%d,test.count()=%d" % (
        trainingData.count(), validationData.count(), testData.count()))

    moviesData = movies.map(lambda fields: (int(fields[0]), fields[1]))
    return trainingData, validationData, testData, moviesData


# ??????
????????,??????????:
- numBlocks is the number of blocks used to parallelize computation (set to -1 to auto-configure).  
- rank is the number of latent factors in the model.  
- iterations is the number of iterations to run.  
- lambda specifies the regularization parameter in ALS.  
- implicitPrefs specifies whether to use the explicit feedback ALS variant or one adapted for implicit feedback data.  
- alpha is a parameter applicable to the implicit feedback variant of ALS that governs the baseline confidence in preference observations.

In [ ]:
# ????,??,?????????,????????????????,??
# ?????????????????????,??????????????
# train????lambda_?????,blocks?????,???-1???????
# iterations?????,rank????user?movies???????????,
# rank???lambda??????????,??????????
def trainModel(trainingData, validationData, testData, iterations=5, lambda_=0.01, blocks=-1):
    ranks = [8, 12]
    lambdas = [1.0, 10.0]
    numIters = [10, 20]
    bestModel = None
    bestValidationRmse = float("inf")
    bestRank = 0
    bestLambda = -1.0
    bestNumIter = -1

    # ??model?data?????????(Mean Squared Error)
    def computeRmse(model, data):
        newData = data.map(lambda r: (r[0], r[1]))  # ???????(??,??),?????????
        predictions = model.predictAll(newData).map(lambda r: ((r[0], r[1]), r[2]))  # ?????(??,??,?????)
        ratesAndPreds = data.map(lambda r: ((int(r[0]), int(r[1])), float(r[2]))).join(
            predictions)  # ??(??,??)??????,????(??,??)?(??,?????)
        return ratesAndPreds.map(lambda r: (r[1][0] - r[1][1]) ** 2).mean()  # ??????????????

    for rank, lmbda, numIter in itertools.product(ranks, lambdas, numIters):  # ????,??2*2*2=8?????
        model = ALS.train(trainingData, rank, numIter, lmbda, blocks)  # ??train???????ALS????,??????????
        validationRmse = computeRmse(model, validationData)  # ?????,?????????????????
        print "RMSE (validation) = %f for the model trained with " % validationRmse + \
              "rank = %d, lambda = %.1f, and numIter = %d." % (rank, lmbda, numIter)
        if (validationRmse < bestValidationRmse):  # ?????????????,???????
            bestModel = model
            bestValidationRmse = validationRmse
            bestRank = rank
            bestLambda = lmbda
            bestNumIter = numIter

    testRmse = computeRmse(bestModel, testData)  # ????????,????????

    print "The best model was trained with rank = %d and lambda = %.1f, " % (bestRank, bestLambda) \
          + "and numIter = %d, and its RMSE on the test set is %f." % (bestNumIter, testRmse)
    return bestModel


In [ ]:
# ???????ALS??????user_id?movies?????????
def predict(model, moviesData, rating, user_id):
    rawMoviesData = dict(moviesData.collect())
    myRateMovieIdsRDD = rating.filter(lambda x: int(x[0]) == user_id). \
        map(lambda x: x[1]).collect()  # ??????????????,???????????id
    myRateMovieIds = set(myRateMovieIdsRDD)  # ????
    candidates = sc.parallelize(
        [m for m in rawMoviesData if m not in myRateMovieIds])  # ????????????????????,???????????????,?????
    predictions = model.predictAll(
        candidates.map(lambda x: (user_id, x))).collect()  # ??(userId,movieId)?,????????,???????
    recommendations = sorted(predictions, key=lambda x: x[2], reverse=True)[:50]  # ???????,??50?????
    print "Movies recommended for you:"
    for i in xrange(len(recommendations)):  # ??????
        print ("%2d: %s" % (i + 1, rawMoviesData[recommendations[i][1]])).encode('ascii', 'ignore')


In [ ]:
# ???????
if __name__ == '__main__':
    sc = initSpark()  # ???Spark???
    trainingData, validationData, testData, moviesData = parseData(sc, "file:///root/notebook/data/ratings.dat",
                                                                   "file:///root/notebook/data/movies.dat")  # ???????
    model = trainModel(trainingData, validationData, testData)  # ??????
    predict(model, moviesData, testData, 1)  # ????????????????
    sc.stop()  # ????Spark???

In [ ]:
# ???????????,????????

def persistModel(model, sparkContext, modelPath):
    model.save(sparkContext, modelPath)


def loadModel(sparkContext, modelPath):
    from pyspark.mllib.recommendation import MatrixFactorizationModel
    return MatrixFactorizationModel.load(sparkContext, modelPath)


if __name__ == '__main__':
    import os

    rawPath = "/root/notebook/data/als_model"
    modelPath = "file://%s" % rawPath
    sc = initSpark()  # ???Spark???
    if os.path.exists(rawPath):
        model = loadModel(sc, modelPath)
        print("load model %s from %s" % (model, rawPath))
    else:
        trainingData, validationData, testData, moviesData = parseData(sc, "file:///root/notebook/data/ratings.dat",
                                                                       "file:///root/notebook/data/movies.dat")  # ???????
        model = trainModel(trainingData, validationData, testData)  # ??????
    predict(model, moviesData, testData, 1)  # ????????????????
    if not os.path.exists(rawPath):
        persistModel(model, sc, modelPath)  # ????,????????
        print("save model %s to %s" % (model, rawPath))
    sc.stop()  # ????Spark???
